# 04 — Broadcasting in Depth

In the previous notebook, we introduced **broadcasting** as a way for PyTorch to perform operations on tensors with different but compatible shapes.

Broadcasting is extremely important in deep learning because it appears in:

- Bias addition
- Feature scaling
- Normalization
- Batch operations
- Image-channel operations
- Attention masks
- Efficient tensor computations

## In this notebook, we will learn:

1. Broadcasting intuition
2. Broadcasting rules
3. Right-to-left shape comparison
4. Scalar broadcasting
5. Vector-to-matrix broadcasting
6. Higher-dimensional broadcasting
7. `torch.broadcast_shapes()`
8. `torch.broadcast_tensors()`
9. `expand()`
10. `repeat()`
11. `expand()` vs `repeat()`
12. Broadcasting with batches
13. Bias addition in neural networks
14. Broadcasting with images
15. Common broadcasting errors
16. Deep-learning broadcasting exercises

## Main Goal

The main goal is to look at two tensor shapes and answer:

> **Can these tensors broadcast together? If yes, what will the output shape be?**

Always try to predict the output shape before running the code.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)

# 1. Broadcasting Intuition

Broadcasting allows PyTorch to perform **element-wise operations** on tensors whose shapes are different but compatible.

Consider:

$$
X =
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
4 & 5 & 6 \\
\hline
\end{array}
$$

Shape:

$$
\boxed{(2,\ 3)}
$$

and:

$$
y =
\begin{array}{|c|c|c|}
\hline
10 & 20 & 30 \\
\hline
\end{array}
$$

Shape:

$$
\boxed{(3)}
$$

When we calculate `X + y`, PyTorch broadcasts the vector across the rows.

Conceptually, PyTorch behaves as if we had:

$$
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
4 & 5 & 6 \\
\hline
\end{array}
+
\begin{array}{|c|c|c|}
\hline
10 & 20 & 30 \\
\hline
10 & 20 & 30 \\
\hline
\end{array}
$$

Result:

$$
\begin{array}{|c|c|c|}
\hline
11 & 22 & 33 \\
\hline
14 & 25 & 36 \\
\hline
\end{array}
$$

> Broadcasting avoids writing manual loops or manually copying the smaller tensor.

In [ ]:
X = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])

y = torch.tensor([10, 20, 30])

result = X + y

print("X shape:", X.shape)
print("y shape:", y.shape)
print("result shape:", result.shape)
print("\nResult:")
print(result)

# 2. Broadcasting Rules

PyTorch compares shapes **from right to left**.

For each pair of dimensions, the dimensions are compatible if:

1. They are equal, or
2. One of them is `1`, or
3. One tensor does not have that dimension

If all compared dimensions satisfy these rules, broadcasting can occur.

## Rule Example — Equal Dimensions

Consider:

$$
(2,\ 3)
$$

and:

$$
(3)
$$

Align from the right:

$$
\begin{array}{cc}
2 & 3 \\
  & 3
\end{array}
$$

The rightmost dimensions are equal:

$$
3 = 3
$$

Therefore:

$$
(2,3) + (3) \rightarrow \boxed{(2,3)}
$$

## Rule Example — A Dimension Is 1

Consider:

$$
(4,\ 1)
$$

and:

$$
(1,\ 5)
$$

Compare from the right:

$$
\begin{array}{|c|c|c|c|}
\hline
\textbf{Position} & \textbf{A} & \textbf{B} & \textbf{Compatible?} \\
\hline
\text{Rightmost} & 1 & 5 & \text{Yes} \\
\hline
\text{Next} & 4 & 1 & \text{Yes} \\
\hline
\end{array}
$$

Output shape:

$$
\boxed{(4,\ 5)}
$$

In [ ]:
a = torch.randn(4, 1)
b = torch.randn(1, 5)

c = a + b

print("a shape:", a.shape)
print("b shape:", b.shape)
print("result shape:", c.shape)

## Rule Example — Missing Dimension

Consider:

$$
(2,\ 3,\ 4)
$$

and:

$$
(4)
$$

Align them from the right:

$$
\begin{array}{ccc}
2 & 3 & 4 \\
  &   & 4
\end{array}
$$

The missing dimensions do not create a conflict.

Therefore:

$$
(2,3,4) + (4) \rightarrow \boxed{(2,3,4)}
$$

In [ ]:
a = torch.randn(2, 3, 4)
b = torch.randn(4)

print("Result shape:", (a + b).shape)

# 3. When Broadcasting Fails

Consider:

$$
(2,\ 3)
$$

and:

$$
(2)
$$

Align from the right:

$$
\begin{array}{cc}
2 & 3 \\
  & 2
\end{array}
$$

The rightmost dimensions are:

$$
3 \neq 2
$$

Neither dimension is `1`.

Therefore the shapes are **not broadcast-compatible**.

In [ ]:
a = torch.randn(2, 3)
b = torch.randn(2)

print("a shape:", a.shape)
print("b shape:", b.shape)

# Uncomment to see the error:
# print(a + b)

# 4. Right-to-Left Shape Comparison

This is the most important practical skill for broadcasting.

Consider:

$$
A.shape = (8,\ 1,\ 6,\ 1)
$$

and:

$$
B.shape = (7,\ 1,\ 5)
$$

Align them from the right:

$$
\begin{array}{cccc}
8 & 1 & 6 & 1 \\
  & 7 & 1 & 5
\end{array}
$$

Compare:

$$
\begin{array}{|c|c|c|c|}
\hline
\textbf{Position} & \textbf{A} & \textbf{B} & \textbf{Result} \\
\hline
\text{Rightmost} & 1 & 5 & 5 \\
\hline
\text{Next} & 6 & 1 & 6 \\
\hline
\text{Next} & 1 & 7 & 7 \\
\hline
\text{Leftmost} & 8 & \text{Missing} & 8 \\
\hline
\end{array}
$$

So the output shape is:

$$
\boxed{(8,\ 7,\ 6,\ 5)}
$$

In [ ]:
A = torch.randn(8, 1, 6, 1)
B = torch.randn(7, 1, 5)

C = A + B

print("A shape:", A.shape)
print("B shape:", B.shape)
print("C shape:", C.shape)

# 5. Scalar Broadcasting

A scalar has shape:

$$
\boxed{()}
$$

A scalar can broadcast across the elements of a tensor.

Suppose:

$$
X =
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
4 & 5 & 6 \\
\hline
\end{array}
$$

Adding the scalar `10` gives:

$$
\begin{array}{|c|c|c|}
\hline
11 & 12 & 13 \\
\hline
14 & 15 & 16 \\
\hline
\end{array}
$$

In [ ]:
X = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])

print(X + 10)

# 6. Vector-to-Matrix Broadcasting

Suppose:

$$
X.shape = (2,\ 3)
$$

and:

$$
v.shape = (3)
$$

The vector aligns with the last dimension:

$$
(2,3) + (3) \rightarrow \boxed{(2,3)}
$$

This is equivalent to adding the same vector to every row.

In [ ]:
X = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])

v = torch.tensor([100, 200, 300])

print(X + v)

## Column-Vector Broadcasting

Suppose:

$$
X =
\begin{array}{|c|c|c|c|}
\hline
1 & 2 & 3 & 4 \\
\hline
5 & 6 & 7 & 8 \\
\hline
9 & 10 & 11 & 12 \\
\hline
\end{array}
$$

and:

$$
v =
\begin{array}{|c|}
\hline
10 \\
\hline
20 \\
\hline
30 \\
\hline
\end{array}
$$

Shapes:

$$
X.shape=(3,4)
$$

$$
v.shape=(3,1)
$$

The `1` can expand across the columns.

In [ ]:
X = torch.tensor([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12]
])

v = torch.tensor([[10], [20], [30]])

print("X shape:", X.shape)
print("v shape:", v.shape)
print("\nX + v:")
print(X + v)

# 7. Higher-Dimensional Broadcasting

Suppose:

$$
A.shape=(5,\ 1,\ 7)
$$

and:

$$
B.shape=(1,\ 6,\ 1)
$$

Compare dimensions:

$$
\begin{array}{ccc}
5 & 1 & 7 \\
1 & 6 & 1
\end{array}
$$

The output shape is:

$$
\boxed{(5,\ 6,\ 7)}
$$

In [ ]:
A = torch.randn(5, 1, 7)
B = torch.randn(1, 6, 1)

C = A + B

print("A shape:", A.shape)
print("B shape:", B.shape)
print("C shape:", C.shape)

# 8. Checking Shapes With `torch.broadcast_shapes()`

PyTorch provides `torch.broadcast_shapes()` to determine the shape that compatible inputs would broadcast to.

In [ ]:
print(torch.broadcast_shapes((5, 1, 7), (1, 6, 1)))

In [ ]:
shape = torch.broadcast_shapes(
    (3, 1, 5),
    (1, 4, 1),
    (5,)
)

print("Broadcasted shape:", shape)

# 9. Inspecting Broadcasting With `torch.broadcast_tensors()`

`torch.broadcast_tensors()` returns broadcasted views of compatible tensors.

This is useful when you want to inspect the logical expansion directly.

In [ ]:
a = torch.tensor([[1], [2], [3]])
b = torch.tensor([[10, 20, 30, 40]])

a_b, b_b = torch.broadcast_tensors(a, b)

print("a shape:", a.shape)
print("b shape:", b.shape)
print("broadcasted a shape:", a_b.shape)
print("broadcasted b shape:", b_b.shape)

print("\nBroadcasted a:")
print(a_b)

print("\nBroadcasted b:")
print(b_b)

# 10. Expanding Tensors With `expand()`

`expand()` lets dimensions of size `1` behave as larger dimensions without normally copying the underlying values.

Suppose:

$$
x =
\begin{array}{|c|c|c|}
\hline
10 & 20 & 30 \\
\hline
\end{array}
$$

Shape:

$$
\boxed{(1,3)}
$$

Using `x.expand(4,3)` gives a view that behaves like:

$$
\begin{array}{|c|c|c|}
\hline
10 & 20 & 30 \\
\hline
10 & 20 & 30 \\
\hline
10 & 20 & 30 \\
\hline
10 & 20 & 30 \\
\hline
\end{array}
$$

In [ ]:
x = torch.tensor([[10, 20, 30]])

expanded = x.expand(4, 3)

print("Original shape:", x.shape)
print("Expanded shape:", expanded.shape)
print(expanded)

## Rules for `expand()`

A dimension can be enlarged by `expand()` when its original size is `1`.

Valid:

$$
(1,3) \rightarrow (5,3)
$$

Not valid for the first dimension:

$$
(2,3) \rightarrow (5,3)
$$

because `2` is not `1`.

Inside `expand()`, `-1` means:

> Keep this dimension unchanged.

In [ ]:
x = torch.randn(1, 3)

y = x.expand(5, -1)

print("Original shape:", x.shape)
print("Expanded shape:", y.shape)

# 11. Repeating Tensors With `repeat()`

`repeat()` creates repeated values according to the requested repetition pattern.

Suppose:

$$
x =
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
\end{array}
$$

Using `x.repeat(4,1)` gives:

$$
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
1 & 2 & 3 \\
\hline
1 & 2 & 3 \\
\hline
1 & 2 & 3 \\
\hline
\end{array}
$$

In [ ]:
x = torch.tensor([[1, 2, 3]])

repeated = x.repeat(4, 1)

print("Original shape:", x.shape)
print("Repeated shape:", repeated.shape)
print(repeated)

## Repeating Across More Than One Dimension

Suppose:

$$
x =
\begin{array}{|c|c|}
\hline
1 & 2 \\
\hline
\end{array}
$$

Using `x.repeat(3,2)` repeats the row 3 times and repeats the values across the second dimension.

In [ ]:
x = torch.tensor([[1, 2]])

y = x.repeat(3, 2)

print(y)
print("Shape:", y.shape)

# 12. `expand()` vs `repeat()`

Both can make values appear multiple times, but they are not the same.

$$
\begin{array}{|c|c|c|}
\hline
\textbf{Property} & \textbf{expand()} & \textbf{repeat()} \\
\hline
\text{Usually copies underlying data} & \text{No} & \text{Yes} \\
\hline
\text{Memory efficient} & \text{Yes} & \text{Less efficient} \\
\hline
\text{Requires expandable size-1 dimension} & \text{Yes} & \text{No} \\
\hline
\text{Useful for broadcasting-like views} & \text{Yes} & \text{Sometimes} \\
\hline
\end{array}
$$

Use `expand()` when you only need a broadcast-like view.

Use `repeat()` when you truly need repeated copies of values.

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0]])

expanded = x.expand(4, 3)
repeated = x.repeat(4, 1)

print("Expanded shape:", expanded.shape)
print("Repeated shape:", repeated.shape)

print("Expanded storage size:", expanded.untyped_storage().nbytes(), "bytes")
print("Repeated storage size:", repeated.untyped_storage().nbytes(), "bytes")

# 13. Broadcasting With Batches

Suppose we have a batch of 4 samples, each with 3 features.

$$
X.shape=(4,3)
$$

A feature-wise scaling vector has shape:

$$
scale.shape=(3)
$$

Then:

$$
(4,3) * (3) \rightarrow \boxed{(4,3)}
$$

The same three scaling values are applied to every sample in the batch.

In [ ]:
X = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
    [10.0, 11.0, 12.0]
])

scale = torch.tensor([1.0, 10.0, 100.0])

print(X * scale)

# 14. Bias Addition in Neural Networks

Bias addition is a classic example of broadcasting.

Suppose a linear layer produces:

$$
XW.shape=(32,10)
$$

where:

- `32` = batch size
- `10` = output features

The bias has shape:

$$
b.shape=(10)
$$

Then:

$$
(32,10) + (10) \rightarrow \boxed{(32,10)}
$$

The same bias vector is added to every sample in the batch.

In [ ]:
batch_output = torch.randn(32, 10)
bias = torch.randn(10)

result = batch_output + bias

print("Batch output shape:", batch_output.shape)
print("Bias shape:", bias.shape)
print("Result shape:", result.shape)

## Visual Bias Example

Suppose:

$$
XW =
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
4 & 5 & 6 \\
\hline
\end{array}
$$

and:

$$
b =
\begin{array}{|c|c|c|}
\hline
10 & 20 & 30 \\
\hline
\end{array}
$$

Then:

$$
XW+b =
\begin{array}{|c|c|c|}
\hline
11 & 22 & 33 \\
\hline
14 & 25 & 36 \\
\hline
\end{array}
$$

# 15. Broadcasting With Image Tensors

PyTorch image batches commonly have shape:

$$
(N,\ C,\ H,\ W)
$$

For example:

$$
(16,\ 3,\ 224,\ 224)
$$

means:

$$
\begin{array}{|c|c|}
\hline
16 & \text{Batch size} \\
\hline
3 & \text{Channels} \\
\hline
224 & \text{Height} \\
\hline
224 & \text{Width} \\
\hline
\end{array}
$$

## Adding One Value Per Channel

Suppose the image batch has shape:

$$
(16,3,224,224)
$$

A per-channel value must align with the channel dimension.

One useful shape is:

$$
(1,3,1,1)
$$

Then:

$$
(16,3,224,224) + (1,3,1,1)
\rightarrow
\boxed{(16,3,224,224)}
$$

In [ ]:
images = torch.randn(16, 3, 224, 224)
channel_bias = torch.tensor([0.1, 0.2, 0.3]).reshape(1, 3, 1, 1)

result = images + channel_bias

print("Images shape:", images.shape)
print("Channel bias shape:", channel_bias.shape)
print("Result shape:", result.shape)

# 16. Image Normalization With Broadcasting

Image normalization often uses one mean and standard deviation per channel.

For RGB data:

$$
mean.shape=(1,3,1,1)
$$

$$
std.shape=(1,3,1,1)
$$

The normalization operation is:

$$
X_{normalized} = \frac{X-mean}{std}
$$

Broadcasting applies each channel's mean and standard deviation across:

- Every image
- Every row
- Every column

In [ ]:
images = torch.randn(8, 3, 64, 64)

mean = torch.tensor([0.5, 0.4, 0.3]).reshape(1, 3, 1, 1)
std = torch.tensor([0.2, 0.25, 0.3]).reshape(1, 3, 1, 1)

normalized = (images - mean) / std

print("Images shape:", images.shape)
print("Mean shape:", mean.shape)
print("Std shape:", std.shape)
print("Normalized shape:", normalized.shape)

# 17. Grayscale Ultrasound Example

Suppose a grayscale ultrasound batch has shape:

$$
\boxed{(16,1,256,256)}
$$

If we want to subtract one mean value per channel, we can use:

$$
mean.shape=(1,1,1,1)
$$

The single value broadcasts over:

- 16 images
- 1 channel
- 256 rows
- 256 columns

In [ ]:
ultrasound = torch.randn(16, 1, 256, 256)
mean = torch.tensor([0.45]).reshape(1, 1, 1, 1)

centered = ultrasound - mean

print("Ultrasound shape:", ultrasound.shape)
print("Mean shape:", mean.shape)
print("Centered shape:", centered.shape)

# 18. Broadcasting a Spatial Mask

Suppose an image batch has shape:

$$
(8,3,64,64)
$$

and a spatial mask has shape:

$$
(64,64)
$$

The mask aligns with the last two dimensions.

Therefore:

$$
(8,3,64,64) * (64,64)
\rightarrow
\boxed{(8,3,64,64)}
$$

The same spatial mask is applied to every image and every channel.

In [ ]:
images = torch.randn(8, 3, 64, 64)
mask = torch.ones(64, 64)

masked = images * mask

print("Images shape:", images.shape)
print("Mask shape:", mask.shape)
print("Result shape:", masked.shape)

# 19. Using `unsqueeze()` to Make Broadcasting Possible

Sometimes two tensors do not broadcast because a dimension is in the wrong position.

Suppose:

$$
X.shape=(3,4)
$$

and:

$$
v.shape=(3)
$$

If we want one value per **row**, `(3)` does not align with the row dimension correctly.

We can convert:

$$
(3) \rightarrow (3,1)
$$

using:

`v.unsqueeze(1)`

In [ ]:
X = torch.tensor([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12]
])

v = torch.tensor([10, 20, 30])

v_column = v.unsqueeze(1)

print("X shape:", X.shape)
print("v shape:", v.shape)
print("v after unsqueeze:", v_column.shape)
print("\nResult:")
print(X + v_column)

# 20. Broadcasting and Reduction Operations

Reduction operations often create shapes that are intended for later broadcasting.

Suppose:

$$
X.shape=(4,3)
$$

If we calculate:

`X.mean(dim=1)`

we get:

$$
(4)
$$

But subtracting `(4)` from `(4,3)` fails because the final dimensions are `3` and `4`.

If we use:

`X.mean(dim=1, keepdim=True)`

we get:

$$
(4,1)
$$

Now subtraction works:

$$
(4,3)-(4,1) \rightarrow (4,3)
$$

In [ ]:
X = torch.randn(4, 3)

row_mean = X.mean(dim=1, keepdim=True)
centered = X - row_mean

print("X shape:", X.shape)
print("Row mean shape:", row_mean.shape)
print("Centered shape:", centered.shape)

# 21. A Powerful Pattern: `keepdim=True`

When you plan to use a reduced result in another operation with the original tensor, `keepdim=True` is often useful.

Example:

$$
X.shape=(32,128)
$$

Mean across features:

$$
X.mean(dim=1,keepdim=True).shape=(32,1)
$$

Then:

$$
(32,128)-(32,1) \rightarrow (32,128)
$$

In [ ]:
X = torch.randn(32, 128)

mean = X.mean(dim=1, keepdim=True)
centered = X - mean

print("X shape:", X.shape)
print("Mean shape:", mean.shape)
print("Centered shape:", centered.shape)

# 22. Common Broadcasting Errors

## Mistake 1 — Comparing Shapes From the Left

Broadcasting compares dimensions from **right to left**, not left to right.

## Mistake 2 — Confusing `(3)` With `(3,1)`

These shapes behave differently:

$$
(3)
$$

is a 1D vector.

$$
(3,1)
$$

is a 2D column-shaped tensor.

## Mistake 3 — Forgetting the Batch Dimension

A per-channel image value often needs shape:

$$
(1,C,1,1)
$$

rather than simply `(C)`.

## Mistake 4 — Using `repeat()` When Broadcasting Is Enough

Unnecessary copies waste memory.

## Mistake 5 — Forgetting `keepdim=True`

A reduction may remove a dimension needed for later broadcasting.

# 23. Broadcasting Debugging Checklist

When broadcasting fails, inspect:

- `x.shape`
- `y.shape`
- The meaning of every dimension

Then:

1. Write the shapes underneath each other
2. Align them from the right
3. Compare one dimension at a time
4. Check whether dimensions are equal or one is `1`
5. Decide the expected output shape
6. Use `unsqueeze()` if a dimension is missing in the wrong place
7. Use `reshape()` when you need a specific broadcast-friendly shape
8. Consider `keepdim=True` after reductions

In [ ]:
def inspect_shapes(a, b):
    print("a shape:", a.shape)
    print("b shape:", b.shape)
    try:
        print("Broadcasted shape:", torch.broadcast_shapes(a.shape, b.shape))
    except RuntimeError as e:
        print("Not broadcast-compatible")
        print("Error:", e)

inspect_shapes(torch.randn(4, 3), torch.randn(3))

# 24. Shape Reasoning Examples

Try to predict each result before running code.

$$
\begin{array}{|c|c|c|}
\hline
\textbf{Shape A} & \textbf{Shape B} & \textbf{Broadcasted Shape} \\
\hline
(4,3) & (3) & (4,3) \\
\hline
(4,1) & (3) & (4,3) \\
\hline
(2,3,4) & (4) & (2,3,4) \\
\hline
(5,1,7) & (1,6,1) & (5,6,7) \\
\hline
(8,3,32,32) & (32,32) & (8,3,32,32) \\
\hline
\end{array}
$$

In [ ]:
examples = [
    ((4, 3), (3,)),
    ((4, 1), (3,)),
    ((2, 3, 4), (4,)),
    ((5, 1, 7), (1, 6, 1)),
    ((8, 3, 32, 32), (32, 32)),
]

for a, b in examples:
    print(a, "+", b, "->", torch.broadcast_shapes(a, b))

# 25. Practice Exercises

Try these without looking at the solutions.

## Exercise 1

Can these shapes broadcast?

$$
(5,3)
$$

and:

$$
(3)
$$

If yes, give the output shape.

## Exercise 2

Can these shapes broadcast?

$$
(5,3)
$$

and:

$$
(5)
$$

Explain why or why not.

## Exercise 3

Predict the result shape:

$$
(4,1) + (1,6)
$$

## Exercise 4

Predict the result shape:

$$
(2,1,5) + (3,1)
$$

## Exercise 5

Create a tensor of shape `(1,4)` and use `expand()` to make it behave as shape `(10,4)`.

## Exercise 6

Create a tensor of shape `(1,4)` and use `repeat()` to create shape `(10,4)`.

## Exercise 7

For image data of shape:

$$
(32,3,224,224)
$$

what shape should a per-channel mean tensor have so that subtraction clearly broadcasts over the batch and spatial dimensions?

## Exercise 8

A tensor has shape `(8,5)`.

Calculate the mean of every row while keeping a broadcastable shape for subtracting the mean from the original tensor.

## Exercise 9

A batch has shape `(64,128)` and a bias has shape `(128)`.

What is the output shape after addition?

## Exercise 10

A grayscale image batch has shape `(16,1,256,256)` and a mask has shape `(256,256)`.

What is the output shape after multiplication?

# 26. Deep-Learning Broadcasting Challenges

Predict before running code.

## Challenge 1

$$
X.shape=(32,128)
$$

$$
b.shape=(128)
$$

What is `(X + b).shape`?

## Challenge 2

$$
X.shape=(16,3,64,64)
$$

$$
mean.shape=(1,3,1,1)
$$

What is `(X - mean).shape`?

## Challenge 3

$$
X.shape=(8,10,32)
$$

$$
v.shape=(32)
$$

Can they broadcast?

## Challenge 4

$$
X.shape=(8,10,32)
$$

$$
v.shape=(10)
$$

Can they broadcast directly?

If not, what shape could `v` be reshaped to if you want one value per middle dimension?

## Challenge 5

$$
X.shape=(4,3)
$$

After:

`mean = X.mean(dim=1)`

what is `mean.shape`?

Will `X - mean` work directly?

How can you fix it?

# 27. Exercise Solutions

In [ ]:
# Exercise 1
print("Exercise 1:", torch.broadcast_shapes((5, 3), (3,)))

# Exercise 2
try:
    print(torch.broadcast_shapes((5, 3), (5,)))
except RuntimeError:
    print("Exercise 2: Not broadcast-compatible")

# Exercise 3
print("Exercise 3:", torch.broadcast_shapes((4, 1), (1, 6)))

# Exercise 4
print("Exercise 4:", torch.broadcast_shapes((2, 1, 5), (3, 1)))

# Exercise 5
x = torch.randn(1, 4)
print("Exercise 5:", x.expand(10, 4).shape)

# Exercise 6
x = torch.randn(1, 4)
print("Exercise 6:", x.repeat(10, 1).shape)

# Exercise 7
mean = torch.randn(1, 3, 1, 1)
print("Exercise 7 mean shape:", mean.shape)

# Exercise 8
X = torch.randn(8, 5)
row_mean = X.mean(dim=1, keepdim=True)
print("Exercise 8 row mean shape:", row_mean.shape)
print("Exercise 8 centered shape:", (X - row_mean).shape)

# Exercise 9
X = torch.randn(64, 128)
b = torch.randn(128)
print("Exercise 9:", (X + b).shape)

# Exercise 10
images = torch.randn(16, 1, 256, 256)
mask = torch.randn(256, 256)
print("Exercise 10:", (images * mask).shape)

# 28. Challenge Solutions

## Challenge 1

$$
(32,128)+(128) \rightarrow \boxed{(32,128)}
$$

## Challenge 2

$$
(16,3,64,64)+(1,3,1,1) \rightarrow \boxed{(16,3,64,64)}
$$

## Challenge 3

`(8,10,32)` and `(32)` are compatible because the rightmost dimensions match.

Output:

$$
\boxed{(8,10,32)}
$$

## Challenge 4

`(8,10,32)` and `(10)` do not broadcast directly because the rightmost dimensions are `32` and `10`.

To apply one value per middle dimension, reshape the vector to:

$$
\boxed{(1,10,1)}
$$

## Challenge 5

`X.mean(dim=1)` has shape:

$$
(4)
$$

`X - mean` does not broadcast correctly with `(4,3)`.

Use:

`X.mean(dim=1, keepdim=True)`

which has shape:

$$
(4,1)
$$

# 29. Key Takeaways

In this notebook, we learned:

- Broadcasting intuition
- Broadcasting rules
- Right-to-left shape comparison
- Scalar broadcasting
- Vector-to-matrix broadcasting
- Higher-dimensional broadcasting
- `torch.broadcast_shapes()`
- `torch.broadcast_tensors()`
- `expand()`
- `repeat()`
- The difference between `expand()` and `repeat()`
- Broadcasting across batches
- Bias addition
- Image-channel broadcasting
- Spatial mask broadcasting
- Using `unsqueeze()` to create broadcast-friendly shapes
- Using `keepdim=True` after reductions

The most important rule is:

> **Compare shapes from right to left. Dimensions must be equal, one must be 1, or one dimension must be missing.**

# 30. Check Your Understanding

Before moving forward, make sure you can answer these questions without searching:

1. What is broadcasting?
2. Why is broadcasting useful?
3. From which side are dimensions compared?
4. What three situations make dimensions compatible?
5. Why can `(4,3)` and `(3)` broadcast?
6. Why can `(4,3)` and `(4)` not normally broadcast?
7. What does `expand()` do?
8. What does `repeat()` do?
9. What is the main difference between `expand()` and `repeat()`?
10. Why is bias addition a broadcasting operation?
11. Why is `(1,C,1,1)` useful for image-channel operations?
12. How can `unsqueeze()` help broadcasting?
13. Why is `keepdim=True` useful after reductions?
14. What shape results from `(5,1,7)` and `(1,6,1)`?
15. How would you debug a broadcasting error?

# Next Notebook

# 05 — Matrix Multiplication

In the next notebook, we will study:

- Element-wise multiplication vs matrix multiplication
- Dot product intuition
- Matrix multiplication rules
- Inner-dimension rule
- Output-shape reasoning
- `torch.matmul()`
- `@` operator
- Matrix-vector multiplication
- Matrix-matrix multiplication
- Batched matrix multiplication
- `torch.bmm()`
- Transpose and matrix multiplication
- Neural-network linear layers from first principles
- Common matrix multiplication errors
- Deep-learning shape exercises